# Navier–Stokes experiments

This notebook is the single entry point for the weighting, flow-difficulty, data-ablation, gradient-diagnostic, and data-only experiments. Edit only the **Run controls** cell and run all cells. Each execution launches one W&B run.

For fair comparisons, change one control at a time and use matched seeds:

- **Loss weighting:** compare `SOFTADAPT_BETA = 0.0` (uniform) with a positive beta.
- **Flow difficulty:** vary `FLOW_CASE`.
- **Data ablation:** vary `NUM_DATA_POINTS`.
- **Gradient diagnostics:** set `TRACK_GRADIENT_METRICS = True`.
- **Data-only baseline:** set `RUN_KIND = 'data_only'`.

In [ ]:
%pip install -q wandb accelerate

## Kaggle setup

In [ ]:
import os

base_path = '/kaggle/working'
project_name = 'weight-vs-loss-convergence-pinn'
project_root = os.path.join(base_path, project_name)

if not os.path.exists(project_root):
    os.chdir(base_path)
    !git clone https://github.com/nthday-jpg/weight-vs-loss-convergence-pinn.git
else:
    os.chdir(project_root)
    !git pull

os.environ['PYTHONPATH'] = project_root
%cd {project_root}

from kaggle_secrets import UserSecretsClient
wandb_api_key = UserSecretsClient().get_secret('WANDB_KEY').strip()
if len(wandb_api_key) < 40:
    raise ValueError(
        'Kaggle secret WANDB_KEY is not a valid W&B API key. '
        'Replace it with the key from https://wandb.ai/authorize.'
    )
os.environ['WANDB_API_KEY'] = wandb_api_key

## Run controls

`NUM_DATA_POINTS = 0` means a physics-only PINN (initial and boundary conditions are still used). A positive integer adds that many interior training observations. Supervised data always excludes initial-time and spatial-boundary points.

In [ ]:
RUN_KIND = 'pinn'              # pinn or data_only
FLOW_CASE = 'smooth'           # smooth, steep, or shock
SOFTADAPT_BETA = 0.0           # 0.0 is uniform; use 0.1 for adaptation
NUM_DATA_POINTS = 0            # nonnegative integer (PINN only)
TRACK_GRADIENT_METRICS = False # True for gradient diagnostics
SEED = 42

## Prepare the benchmark data

In [ ]:
import subprocess
import sys

case_presets = {
    'smooth': {
        'nx': 256, 'steps': 200, 't_final': 0.6,
        'mu': 1e-2, 'spatial_scheme': 'central',
    },
    'steep': {
        'nx': 384, 'steps': 250, 't_final': 0.4,
        'mu': 3e-3, 'spatial_scheme': 'rusanov',
    },
    'shock': {
        'nx': 512, 'steps': 300, 't_final': 0.3,
        'mu': 5e-4, 'spatial_scheme': 'rusanov',
    },
}

if RUN_KIND not in {'pinn', 'data_only'}:
    raise ValueError("RUN_KIND must be 'pinn' or 'data_only'")
if FLOW_CASE not in case_presets:
    raise ValueError(f'Unknown FLOW_CASE: {FLOW_CASE}')
if not isinstance(SOFTADAPT_BETA, (int, float)) or SOFTADAPT_BETA < 0:
    raise ValueError('SOFTADAPT_BETA must be a nonnegative number')
if not isinstance(NUM_DATA_POINTS, int) or NUM_DATA_POINTS < 0:
    raise ValueError('NUM_DATA_POINTS must be a nonnegative integer')
if RUN_KIND == 'data_only' and TRACK_GRADIENT_METRICS:
    raise ValueError('Gradient diagnostics apply only to PINN runs')

case = case_presets[FLOW_CASE]
base_name = f'navier_stokes_{FLOW_CASE}'
data_file = os.path.join(project_root, 'navier-stokes', 'data', base_name + '.pt')

subprocess.run([
    sys.executable, 'navier-stokes/data/gen_data.py',
    '--save_dir', 'navier-stokes/data',
    '--base_name', base_name,
    '--flow_case', FLOW_CASE,
    '--spatial_scheme', case['spatial_scheme'],
    '--nx', str(case['nx']),
    '--steps', str(case['steps']),
    '--t_final', str(case['t_final']),
    '--mu', str(case['mu']),
], check=True)

## Build and review the training configuration

In [ ]:
import json

config = {
    'problem': 'navier_stokes',
    'layers': [2, 64, 64, 64, 3],
    'learning_rate': 1e-3,
    'num_steps': 50_000,
    'adapt_every_steps': 100,
    'log_every_steps': 100 if TRACK_GRADIENT_METRICS else 50,
    'batch_size': 8192,
    'seed': SEED,
    'train_time_fraction': 0.7,
    'validation_time_fraction': 0.85,
    'wandb_project': 'loss-weight-convergence',
}

if RUN_KIND == 'pinn':
    config.update({
        'num_data_points': NUM_DATA_POINTS,
        'track_gradient_metrics': TRACK_GRADIENT_METRICS,
        'softadapt_params': {
            'beta': SOFTADAPT_BETA,
            'history_length': 5,
            'normalized': True,
            'loss_weighted': False,
        },
    })
    gradient_tag = '-gradients' if TRACK_GRADIENT_METRICS else ''
    run_label = f'pinn-beta-{SOFTADAPT_BETA:g}-data-{NUM_DATA_POINTS}{gradient_tag}'
else:
    config.update({
        'num_data_points': 'all',
        'loss_terms': ['data'],
        'softadapt_params': {'beta': 0.0},
    })
    run_label = 'data-only-all'

config['wandb_run_name'] = (
    f'navier-stokes-{FLOW_CASE}-{run_label}-seed-{SEED}'
)
os.environ['PINN_CONFIG'] = json.dumps(config)
os.environ['PINN_DATA_FILE'] = data_file

print(json.dumps(config, indent=2))
print(f'\nData: {data_file}')

## Train

A single process keeps adaptive-balancer state deterministic and synchronized.

In [ ]:
!accelerate launch {project_root}/train.py

In [ ]:
from IPython.display import Image, display

plot_name = config['wandb_run_name'] + '_comparison.png'
display(Image(filename=os.path.join(project_root, 'results', plot_name)))
print('Convergence tables and diagnostic plots are available in the W&B run.')